In [28]:
import cobra
from corda import CORDA

import pandas as pd
import random

In [29]:
# load inputs
build_files_path = '/data2/hratch/human_me/build_files/'
full_model = cobra.io.load_json_model('/data2/hratch/human_me/input_files/recon2_2.json')
rmd = pd.read_csv(build_files_path + 'required_metabolic_model_metabolites.csv', index_col = 0)

In [30]:
# load parameters
max_score = 3
frac_reactions = 0.01

# generate reaction confidence scores
n_reactions = len(full_model.reactions)
n_reactions_to_keep = round(frac_reactions*n_reactions)

one_reaction = [r.id for r in full_model.reactions if len(r.genes)==1]
reactions_to_exclude = random.sample(one_reaction, round(len(one_reaction)*.25)) # decreases prob of choosing a reaction with a gene by 4x
reactions_to_exclude += [r.id for r in full_model.reactions if len(r.genes)>1]


population = sorted(set([r.id for r in full_model.reactions]).difference(reactions_to_exclude))
reactions_to_keep = random.sample(population, k = n_reactions_to_keep)

conf = {}
for r in full_model.reactions: 
    if r.id not in reactions_to_keep:
        conf[r.id] = -1
    else:
        conf[r.id] = random.choice(list(range(max_score +1)))
conf["biomass_reaction"] = 3

In [31]:
# do the extraction
opt = CORDA(full_model, conf, met_prod = rmd.index.tolist())
opt.build()
print(opt)

build status: reconstruction complete
Inc. reactions: 635/7844
 - unclear: 3/21
 - exclude: 550/7706
 - low and medium: 6/38
 - high: 76/79



In [32]:
toy_model = opt.cobra_model('toy_model')

In [33]:
cobra.io.save_json_model(model = toy_model, filename = '/data2/hratch/human_me/input_files/toy_model.json')